In [ ]:
# !pip install pytorch-lightning
# !pip install tokenizers
# !pip install pyarabic

In [ ]:
5

In [1]:
from torchaudio.pipelines import WAV2VEC2_LARGE_LV60K, HUBERT_BASE
import torch
import pytorch_lightning as pl

import numpy as np
import os
from torch.utils.data import DataLoader, Dataset
import gc
gc.collect()

/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: libtensorflow_io.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so: undefined symbol: _ZTVN10tenso

3007

In [2]:
import re
import pandas as pd
import numpy as np
import yaml
from yaml import CLoader
import os
from os.path import join
from tqdm import tqdm
import re
from tokenizers import Tokenizer
from pyarabic.trans import normalize_digits

In [3]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = "false"

In [4]:
class TokenHandler:
    def __init__(self, json_path: str, lang='en'):

        self.tok = Tokenizer.from_file(json_path)
        self.tok.enable_padding(pad_id=self.get_id("<PAD>"), pad_token="<PAD>")
        if lang == 'en':
            self.pre = self.english_preprocess
        elif lang == 'ar':
            self.pre = self.arabic_preprocess
        else:
            raise NotImplementedError('This class suports En and Ar language only for now')
    def arabic_preprocess(self, s: str):
        '''Remove non arabic characters and unnecessary spaces.
        @input: string
        @return: cleaned string
        '''
        s = normalize_digits(s, source='all', out='west')
        s = re.sub('\(\s*[ء-ي]*\s*\)+', '', s )
        s = re.sub(r'[?]+', "؟", s)
        s = re.sub('[^\sء-ي؟!.1-9]+', '', s )
        s = re.sub(r'[.]+', ".", s)
        s = re.sub(r'[" "]+', " ", s)
        s = s.rstrip().strip()
        return s
    
    def english_preprocess(self, s: str):
        '''Remove non english characters and unnecessary spaces.
        @input: string
        @return: cleaned string
        '''

        s = re.sub(r"\([\sa-zA-Z]+\)+", " ", s)
        s = re.sub(r"[^\sa-zA-Z0-9?!'.]+", "", s)
        s = re.sub(r'[.]+', ".", s)
        s = re.sub(r'[" "]+', " ", s)

        s = s.rstrip().strip()

        return s
        
    
    def __call__(self, text, length=None):
        text = self.pre(text)
        out = self.tok.encode(text)
        if length is not None:
            out.pad(length, pad_id=self.get_id("<PAD>"), pad_token="<PAD>")
            out.truncate(length)            
        return out.ids
        
    def encode(self, text: str):
        '''@input: text --> single string.
        @return:  ids, tokens
        '''
        text = self.pre(text)
        out = self.tok.encode(text)
        return out
    
    def get_id(self, token: int):
        '''@input: token --> single word 
        @return: id --> int
        '''
        return self.tok.token_to_id(token)
    
    def encode_batch(self, data: list):
        '''@input: data --> list of strings.
        @return:  ids, tokens
        '''
        data = tuple(map(self.pre, data))
        output = self.tok.encode_batch(data)
        return output
    
    def decode(self, ids: list):
        '''@input: ids --> list of int
        @return: text --> single string.
        '''
        return self.tok.decode(ids)
    
    def decode_batch(self, ids: list):
        return self.tok.decode_batch(ids)


In [5]:
en_json = '/kaggle/input/helper-for-s2t/en_tokenizer.json'
ar_json = '/kaggle/input/helper-for-s2t/ar_tokenizer.json'

In [6]:
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass, field

from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from pytorch_lightning.callbacks import Callback, RichProgressBar, ModelCheckpoint, StochasticWeightAveraging
from pytorch_lightning.callbacks.progress.rich_progress import RichProgressBarTheme

import torchmetrics.functional as MF

from rich import print as rprint

In [7]:
class MuSTCDataset(Dataset):
    def __init__(self, data, batch_size):
        super(MuSTCDataset, self).__init__()
        self.files = []
        self.batch_size = batch_size
        for f in data:
            in_dire = [i for i in os.listdir(f) if '.npy' in i]
            in_dire = list(map(lambda x: os.path.join(f, x), in_dire))
            self.files.extend(in_dire)
        self.files = np.array(self.files)

            
    def  __getitem__(self, idx):
        data = self.files[idx]
        data = np.load(data, allow_pickle=True)
        r = np.random.randint(0, 33 - self.batch_size)
        wavex = data[r:r+self.batch_size, :240000].astype(np.float32)
        en    = data[r:r+self.batch_size, 240000:240050].astype(np.int64)
#         ar    = data[r:r+self.batch_size, 240050:].astype(np.int64)

        return torch.from_numpy(wavex), torch.from_numpy(en) #, torch.from_numpy(ar)
        
    def __len__(self):
        return len(self.files)

In [8]:
class DecoderLayer(nn.Module):
    def __init__(self, **kwargs):
        """@params:
                    d_model, nhead, batch_first=True
        """
        super(DecoderLayer, self).__init__()
        self.config = kwargs
        
        self.pre_norm1 = nn.LayerNorm(self.config['d_model'])
        
        self.smha      = nn.MultiheadAttention(embed_dim=self.config['d_model'], num_heads=self.config['nhead'], dropout=0.01, 
                                               bias=True, add_bias_kv=True, batch_first=self.config['batch_first'])

        self.pre_norm2 = nn.LayerNorm(self.config['d_model'])
        
        self.cmha      = nn.MultiheadAttention(embed_dim=self.config['d_model'], num_heads=self.config['nhead'], dropout=0.01, 
                                               bias=True, add_bias_kv=True, batch_first=self.config['batch_first'])
        
        self.ffn       = nn.Sequential(nn.LayerNorm(self.config['d_model']),
                                       nn.Linear(self.config['d_model'], 1024),
                                       nn.ReLU(),
                                       nn.Linear(1024, self.config['d_model']),
                                       nn.ReLU())
        
        
    def forward(self, query, key, query_mask=None, key_mask=None, att_mask=None, need_weights=False, training=False):
        if query.dim() < 3:
            query = query.view(1, -1, self.config['d_model']).contiguous()
            
        if key.dim() < 3:
            key = key.view(1, -1, self.config['d_model']).contiguous()
            
        query = self.pre_norm1(query)
        

        
        att_out, _ = self.smha(query, query, query, key_padding_mask=query_mask, need_weights=False, attn_mask=att_mask)
        att_out = att_out + query
        
        query = self.pre_norm2(query)
        
        att_out, att_weight = self.cmha(query, key, key, key_padding_mask=key_mask, attn_mask=None,
                                        need_weights=need_weights, )
        
        att_out = att_out + query; del query
        
        att_out = att_out + self.ffn(att_out)
        
        if need_weights:
            return att_out, att_weight
        
        return att_out
    
    def init_weights(self):
        
        for p in self.parameters():    
            if p.dim() != 1:
                nn.init.xavier_normal_(p.data, gain=nn.init.calculate_gain('relu'))
            
        for m in self.modules():
            if isinstance(m, (nn.LayerNorm, )):
                m.reset_parameters()
                
    def freeze(self, state=True):
        self.requires_grad_(state)

                
    def get_config(self):
        return self.config



class Decoder(nn.Module):
    def __init__(self, **kwargs):
        '''d_model=512, nhead=4,  batch_first=True, size'''
        super().__init__()
        self.config = kwargs
        assert self.config['d_model'] % self.config['nhead'] == 0, 'd_model should be dvisible by num of heads'
        
        self.dec_stack = nn.ModuleList([DecoderLayer(d_model=self.config['d_model'], nhead=self.config['nhead'],
                                                     batch_first=self.config['batch_first']) 
                                        for _ in range(self.config['size'])])
        
        
    def forward(self, query, key, query_mask=None, key_mask=None, att_mask=None, need_weights=False, training=False):
        if need_weights:
            for layer in self.dec_stack:
                query, atten_weight = layer(query, key, query_mask=query_mask, key_mask=key_mask, att_mask=att_mask, 
                                            need_weights=need_weights, training=training)
            return query, atten_weight
        
        for layer in self.dec_stack:
            query = layer(query, key, query_mask=query_mask, key_mask=key_mask, att_mask=att_mask,
                          need_weights=need_weights, training=training)
            
        return query
    
    def init_weights(self):
        
        for p in self.parameters():
            if p.dim() != 1:
                nn.init.xavier_normal_(p.data, gain=nn.init.calculate_gain('relu'))
            else:
                nn.init.zeros_(p.data)
                
        for m in self.modules():
            if isinstance(m, (nn.LayerNorm,)):
                m.reset_parameters()
                
                
    def freeze(self, state=True):
        
        self.requires_grad_(state)
    

                
    def get_config(self):
        return self.config

class Head(nn.Module):
    def __init__(self, **kwargs):
        '''d_model, voc_size'''
        super().__init__()
        self.config = kwargs
        
        self.layer1 =  nn.Sequential(nn.Linear(self.config['d_model'], self.config['d_model'] * 2),
                                     nn.ReLU(),
                                     nn.Linear(self.config['d_model'] * 2, self.config['voc_size']))

       
    def forward(self, x, **kwargs):
        x = self.layer1(x)
      
        return x
        
    def init_weights(self):
        
        for p in self.parameters():
            if p.dim() != 1:
                nn.init.xavier_normal_(p.data)
            else:
                nn.init.zeros_(p.data)
                
    def freeze(self, state=True):
        self.requires_grad_(state)

                
    def get_config(self):
        return self.config


In [9]:
class Model(pl.LightningModule):  
    
    def __init__(self, lr, decoder_params: dict, head_names: dict, head_params: dict):
        super(Model, self).__init__()
        self.save_hyperparameters()
        
        
        self.enc = HUBERT_BASE.get_model()
        
        self.heads = nn.ModuleDict()
        for h, pad_idx in self.hparams.head_names.items():
            self.heads[h] = nn.ModuleDict({'emp_layer': nn.Embedding(num_embeddings=self.hparams.head_params[h]['voc_size'], 
                                                                     embedding_dim=self.hparams.head_params[h]['d_model'], padding_idx=pad_idx),
                                            'context': Decoder(**self.hparams.decoder_params),
                                            'output_layer': Head(**self.hparams.head_params[h])})
#         self.init_weights()
        
    def pe(self, length: int, depth: int, device, n=10000):
        '''create positionalemppeding matrix
        @params:
                length:  Max number of tokens in as sentence that the model will deal with it during inference.
                depth:   Empeddingdim
        '''
        
        positions = torch.arange(length, device=device).view(-1, 1)    # (seq, 1)  [0, 1, 2, 3 ... length-1]

        depths = torch.arange(depth, device=device).view(1, -1) / depth   # (1, depth) [0 / depth, 1 / depth, 2/depth, 3/depth ... length-1/depth]

        angle_rates = 1 / (n**depths)             # (1, depth)

        angle_rads = positions * angle_rates      # (pos, depth)

        angle_rads[:, 0::2] = torch.sin(angle_rads[:, 0::2])

        # apply cos to odd indices in the array; 2i+1
        angle_rads[:, 1::2] = torch.cos(angle_rads[:, 1::2])
    #         print(angle_rads.shape)
        return angle_rads.float()   
    
    def forward(self, wave, target: dict, training=False, wave_mask=False,
                target_mask=False, dec_mask=False, need_dec_weights=False):
        
        model_output = dict() 
        for h, _ in self.hparams.head_names.items():
            model_output[h] = dict()
        
        with torch.inference_mode():
            wave, _ = self.enc(wave)
    
        for h, _ in self.hparams.head_names.items():
            
            target_head = target[h]
            assert target_head.dim() < 3, f'Head: {h}, target size should be 1D tensor for unbatched and 2D for batched'
            if target_head.dim() < 2:
                target_head = target_head.view(1, -1)
                
                        
            query_mask = None
            if target_mask:
                query_mask = (target_head == self.hparams.head_names.get(h)).to(target_head.device)
            
            if dec_mask:
                att_mask = self.look_ahead_mask(target_head.size(1), target_head.size(1), device=target_head.device)
                
            else: 
                att_mask = None
                
            
            target_head = self.heads[h]['emp_layer'](target_head)
            B, T, D = target_head.size()
            target_head = self.pe(T, D, target_head.device) + target_head
                        
            
            target_head = target_head = self.heads[h]['context'](query=target_head, key=wave, query_mask=query_mask, att_mask=att_mask,
                                              need_weights=need_dec_weights, training=training)

            if need_dec_weights:
                target_head, model_output[h]['attention_weights'] = target_head[0], target_head[1].detach()
            
            model_output[h]['predection'] = self.heads[h]['output_layer'](target_head); del target_head
            
                
        return model_output
    
    def training_step(self, batch, batch_idx):
        
        wave, en = batch
        ground_truth = {'en': en[:, 1:]}
        results = self(wave=wave, target={'en': en[:, :-1]}, training=True, wave_mask=True, 
                       target_mask=True, dec_mask=True)
        loss = 0
        named_loss = dict()
        named_loss[f'Loss'] = loss
        for h, pad_idx in self.hparams.head_names.items():
            preds = results[h]['predection'].transpose(2, 1)
            h_loss = F.cross_entropy(preds, ground_truth[h], reduction='mean', ignore_index=pad_idx, label_smoothing=0.15)
            loss += h_loss
            
            
            named_loss[f'{h}_Acc'] = MF.classification.multiclass_accuracy(preds.detach(), ground_truth[h], ignore_index=pad_idx,
                                                                           num_classes=self.hparams.head_params[h]['voc_size'])
        named_loss[f'Loss'] = loss.detach()
                
        
        self.log_dict(named_loss, on_step=True, on_epoch=False, prog_bar=True, )#sync_dist=True,)
        
        return loss
    
    def validation_step(self, batch, batch_idx):
            
        wave, en = batch
        ground_truth = {'en': en[:, 1:]}
        results = self(wave=wave, target={'en': en[:, :-1]}, training=False, wave_mask=True, 
                       target_mask=True, dec_mask=True)
        loss = 0
        named_loss = dict()
        for h, pad_idx in self.hparams.head_names.items():
            preds = results[h]['predection'].transpose(2, 1)
            h_loss = F.cross_entropy(preds, ground_truth[h], reduction='mean', ignore_index=pad_idx, label_smoothing=0.09)
            loss += h_loss
#             named_loss[f'{h}_{at}_Loss'] = h_loss
            named_loss[f'{h}_val_Acc'] = MF.classification.multiclass_accuracy(preds.detach(), ground_truth[h], ignore_index=pad_idx,
                                                                           num_classes=self.hparams.head_params[h]['voc_size'])
        named_loss['val_loss'] = loss.detach()
        self.log_dict(named_loss, on_step=False, on_epoch=True, prog_bar=True, )#sync_dist=True,)
        
        
        
    def configure_optimizers(self):
        optimizer = torch.optim.SGD([p for n, p in self.named_parameters() if 'enc' not in n ], lr=self.hparams.lr)
#         optimizer = torch.optim.SGD(self.parameters(), lr=self.hparams.lr)

        scheduler = {
            "scheduler": CosineAnnealingWarmRestarts(optimizer, T_0=6, T_mult=2, eta_min=5e-5, verbose=True),
            "interval": "epoch",
            "frequency": 1,}
        return [optimizer], [scheduler]

    
    def look_ahead_mask(self, tgt_len:int, src_len: int, device):
        mask = torch.triu(torch.ones((tgt_len, src_len), device=device), diagonal=1).type(torch.bool)
        
        return mask
    
    def init_weights(self):
        
        for p in self.parameters():    
            if p.dim() != 1:
                nn.init.xavier_normal_(p.data)
            else:
                nn.init.constant_(p.data, 0)
                
        for m in self.modules():
            if isinstance(m, (nn.LayerNorm,)):
                m.reset_parameters()
                
        self.enc = HUBERT_BASE.get_model()
        

In [10]:
class Predictions(Callback):
    def __init__(self, config):
        super().__init__()
        self.tokenizers = dict()
        for k, v in config.items():
            self.tokenizers[k] = TokenHandler(v, k)
        
    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, *args, **kwargs):
        if not batch_idx % 3000:
        
            wave, en = batch
            ground_truth = {'en': en[:, 1:]}
        
            

            with torch.no_grad():
                results = pl_module(wave=wave, target={'en': en[:, :-1]}, training=False, wave_mask=True, 
                       target_mask=True, dec_mask=True)
            pred = ''
            ground = ''
            r = torch.randint(0, en.size(0), (1, )).item()
            for h, pad_idx in pl_module.hparams.head_names.items():

                t = self.tokenizers[h].decode_batch(results[h]['predection'].detach().argmax(-1).tolist())
                j = self.tokenizers[h].decode_batch(ground_truth[h].detach().tolist())
                blue = MF.bleu_score(t, j)
                
                pred += f"'{h} guess: with blue score: {blue:0.4f} ' \n\t {t[r]}\n\n"
                ground += f"'{h}' \n\t {j[r]}\n\t"

            rprint(f'\nGround Truth {batch_idx}: \n\t {ground} \n\n {pred}')
                
    def load_state_dict(self, state_dict):
        self.tokenizers.update(state_dict)

    def state_dict(self):
        return self.tokenizers.copy()
    
en_json = '/kaggle/input/helper-for-s2t/en_tokenizer.json'
ar_json = '/kaggle/input/helper-for-s2t/ar_tokenizer.json'
pred = Predictions({'en': en_json}) 


ckp = ModelCheckpoint(every_n_train_steps=1000, save_last=True, auto_insert_metric_name=False,
                      dirpath='/kaggle/working/lightning_logs/checkpoints')
swa = StochasticWeightAveraging(swa_lrs=1e-5, annealing_epochs=15, swa_epoch_start=5)

In [11]:
train_path = '/kaggle/input'
train_path = [os.path.join(train_path, i) for i in os.listdir(train_path) if 'spliter' in i]
val_path = ['/kaggle/input/spliter-107250117000']

pl.seed_everything(22, workers=True)
worker = 2
accelerator = 'auto'
devices = 'auto'
strategy = 'auto'
epochs = 10000

In [12]:
class DataLightning(pl.LightningDataModule):
    def __init__(self, batch_size):
        super().__init__()
        self.batch_size = batch_size
    def train_dataloader(self):
        train_loader = MuSTCDataset(train_path, self.batch_size)
        
        train_loader = DataLoader(train_loader, batch_size=None,  shuffle=True, pin_memory=True, num_workers=worker, persistent_workers=True)
        return train_loader 
    def val_dataloader(self):
        val_loader = MuSTCDataset(val_path, self.batch_size)
        val_loader = DataLoader(val_loader, batch_size=None,  shuffle=False, pin_memory=True, num_workers=worker, persistent_workers=True,)
        return val_loader


In [13]:
datamoel = DataLightning(8)
decoder_params  = dict(d_model=768, nhead=4, batch_first=True, size=2)

head_params     = dict(en=dict(d_model=768, voc_size=1000))

tokenizers      = dict(en=TokenHandler(en_json, 'en'))

head_names      = dict(en=tokenizers['en'].get_id("<PAD>"))
hyper_parameter = dict(lr=5e-3, decoder_params=decoder_params, 
                        head_names=head_names, head_params=head_params)

model = Model(**hyper_parameter)


In [ ]:
trainer = pl.Trainer(accelerator=accelerator, devices=devices, 
                     max_epochs=epochs,
                     strategy=strategy,
                     num_sanity_val_steps=3,
                     log_every_n_steps=450,
                     callbacks=[ ckp, pred],
                     accumulate_grad_batches=30,
#                      limit_train_batches=12000,
                     gradient_clip_val=50,
                     
                     enable_model_summary=True, enable_checkpointing=True,# benchmark=True, 
                     default_root_dir='/kaggle/working/')

trainer.fit(model, datamoel,) # ckpt_path='/kaggle/working/lightning_logs/checkpoints/last.ckpt')

Epoch 00000: adjusting learning rate of group 0 to 5.0000e-03.


Sanity Checking: 0it [00:00, ?it/s]

/opt/conda/lib/python3.10/site-packages/torch/nn/functional.py:4999: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Training: 0it [00:00, ?it/s]

Ground Truth 0: 
         'en' 
         CO So tornadoes don ' t happen in Massachusetts and I was cleverly standing in the front yard when one 
came over the hill.
         

 'en guess: with blue score: 0.0000 ' 
         tell te te takeble te diff sec re awn re n al alXings Bover moment pro B Bings te L diffim an Bup 3ang 
first P Bryoveraut te through al te first take te knowY al

...

Ground Truth 3000: 
         'en' 
         I should think they ' ll say yes because they ' ll know that they make software that goes on computers and
occasionally works.
         

 'en guess: with blue score: 0.0000 ' 
         And ' the the ' ss the ' a.s ss. 's the.s the............. thes............

Ground Truth 6000: 
         'en' 
         What are the things that we wake up loving to do no matter what whether we ' re paid or we ' re not paid 
the things that people thank us for?
         

 'en guess: with blue score: 0.0000 ' 
         And ' the ' the ' ' the. thes. the. the.... ' s the.... s the....................

Epoch 00005: adjusting learning rate of group 0 to 3.8159e-04.


Validation: 0it [00:00, ?it/s]

Ground Truth 0: 
         'en' 
         He ' ll find Jamie wherever he is you know wherever kids are crushing rocks by a stream or something and 
take him a little message that says Hey dinner ' s
         

 'en guess: with blue score: 0.0000 ' 
         And the s 's the to. the the. the the................... thes.............. s

In [ ]:
!rm -r /kaggle/working/lightning_logs/

In [ ]:
os.listdir('/kaggle/working/')

In [ ]:
enc = HUBERT_BASE.get_model()
for p in enc.modules():
    p.requires_grad_(False)
    break


In [ ]:
for i in p.modules():
    print(i)

In [ ]:
[p for p in enc.modules()]